# Robo-Greeno (3+3) Hexapod — reinforcement learning on Colab GPU

Self-contained notebook. Trains a walking policy for the six-leg hexapod with PPO on a Colab T4 GPU, then renders a video of the trained spider walking.

**Before you run:** click **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Total wallclock on a free T4: about 25–35 minutes for ~1.5M training steps.

What you'll see, in order:
1. The hexapod model loaded in MuJoCo (rest-pose render).
2. A short clip of random-action behavior — baseline before learning.
3. PPO training progress.
4. A side-by-side video of the trained policy walking, with a top-down trajectory plot.

In [ ]:
# 1. Setup. Installs + GPU check.
import os, sys, subprocess, time

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "mujoco", "mediapy", "gymnasium",
                           "stable-baselines3[extra]", "tensorboard"])

os.environ.setdefault("MUJOCO_GL", "egl")

import numpy as np
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import gymnasium as gym
from gymnasium import spaces
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Runtime → Change runtime type → T4 GPU, then Run all.")
DEVICE = "cuda"
print(f"MuJoCo:   {mujoco.__version__}")
print(f"PyTorch:  {torch.__version__}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")

## 1. Build the hexapod model

Six legs (3 left + 3 right) attached to a box chassis. Each leg has 3 hinge joints (coxa + femur + tibia). 18 joint actuators total. Identical kinematic structure to `robo_greeno_3plus3_colab.ipynb`, plus position actuators and ground-contact enabled.

In [ ]:
# 2. Build the model XML programmatically.
CHASSIS_LX, CHASSIS_LY, CHASSIS_LZ = 0.16, 0.10, 0.025
COXA_LEN, FEMUR_LEN, TIBIA_LEN = 0.05, 0.15, 0.20
BODY_Z = 0.22

REST_COXA_DEG = 0.0
REST_FEMUR_DEG = -20.0
REST_TIBIA_DEG = -60.0
ACTION_RANGE_RAD = 0.3

LEG_LAYOUT = [
    ("LF",  0.12, "L"), ("LM",  0.00, "L"), ("LR", -0.12, "L"),
    ("RF",  0.12, "R"), ("RM",  0.00, "R"), ("RR", -0.12, "R"),
]
LEG_NAMES = [n for n, _, _ in LEG_LAYOUT]

def _make_leg(name, attach_x, side):
    if side == "L":
        attach_y, fy = +CHASSIS_LY, +1
        coxa_axis, lift_axis = "0 0 -1", "1 0 0"
    else:
        attach_y, fy = -CHASSIS_LY, -1
        coxa_axis, lift_axis = "0 0 1", "-1 0 0"
    return f"""
      <body name=\"{name}_coxa\" pos=\"{attach_x} {attach_y} 0\">
        <joint name=\"{name}_coxa_j\"  axis=\"{coxa_axis}\" range=\"-45 45\"/>
        <geom  type=\"capsule\" size=\"0.012\" fromto=\"0 0 0  0 {fy*COXA_LEN} 0\" rgba=\"0.30 0.50 0.30 1\"/>
        <body name=\"{name}_femur\" pos=\"0 {fy*COXA_LEN} 0\">
          <joint name=\"{name}_femur_j\" axis=\"{lift_axis}\" range=\"-60 60\"/>
          <geom  type=\"capsule\" size=\"0.011\" fromto=\"0 0 0  0 {fy*FEMUR_LEN} 0\" rgba=\"0.20 0.65 0.40 1\"/>
          <body name=\"{name}_tibia\" pos=\"0 {fy*FEMUR_LEN} 0\">
            <joint name=\"{name}_tibia_j\" axis=\"{lift_axis}\" range=\"-90 30\"/>
            <geom  type=\"capsule\" size=\"0.009\" fromto=\"0 0 0  0 {fy*TIBIA_LEN} 0\" rgba=\"0.10 0.75 0.45 1\"/>
            <site name=\"{name}_foot\" pos=\"0 {fy*TIBIA_LEN} 0\" size=\"0.012\" rgba=\"1 0.4 0 1\"/>
          </body>
        </body>
      </body>"""

_legs_xml = "\n".join(_make_leg(n, x, s) for n, x, s in LEG_LAYOUT)
_acts = []
for n, _, _ in LEG_LAYOUT:
    for jn in (f"{n}_coxa_j", f"{n}_femur_j", f"{n}_tibia_j"):
        _acts.append(f'    <position name="{jn}_act" joint="{jn}" kp="35" forcerange="-3 3"/>')
_actuator_xml = "\n".join(_acts)

MODEL_XML = f"""
<mujoco model=\"robo_greeno_3plus3_rl\">
  <compiler angle=\"degree\" autolimits=\"true\"/>
  <option gravity=\"0 0 -9.81\" timestep=\"0.005\"/>
  <visual><global offwidth=\"1024\" offheight=\"768\"/></visual>
  <default>
    <joint type=\"hinge\" damping=\"0.5\" armature=\"0.005\"/>
    <geom contype=\"1\" conaffinity=\"1\" density=\"500\" friction=\"1.2 0.005 0.0001\"
          solref=\"0.005 1\" solimp=\"0.9 0.95 0.001\"/>
    <site size=\"0.012\"/>
  </default>
  <worldbody>
    <light pos=\"0 -2 2\" dir=\"0 0.5 -1\"/>
    <camera name=\"chase\" pos=\"-0.6 -0.9 0.55\" xyaxes=\"0.83 -0.55 0  0.18 0.27 0.95\" fovy=\"55\"/>
    <camera name=\"side\"  pos=\"0 -1.2 0.35\"   xyaxes=\"1 0 0  0 0.3 1\" fovy=\"50\"/>
    <geom name=\"ground\" type=\"plane\" size=\"10 10 0.05\" rgba=\"0.88 0.88 0.92 1\"/>
    <body name=\"chassis\" pos=\"0 0 {BODY_Z}\">
      <freejoint name=\"root\"/>
      <geom name=\"chassis_g\" type=\"box\" size=\"{CHASSIS_LX} {CHASSIS_LY} {CHASSIS_LZ}\" rgba=\"0.20 0.50 0.30 1\"/>
      {_legs_xml}
    </body>
  </worldbody>
  <actuator>
{_actuator_xml}
  </actuator>
</mujoco>
"""

_test_model = mujoco.MjModel.from_xml_string(MODEL_XML)
print(f"loaded: nq={_test_model.nq}, nu={_test_model.nu}, nbody={_test_model.nbody}")

In [ ]:
# 3. Visualize the rest pose so you can see the model before training.
_d = mujoco.MjData(_test_model)
_d.qpos[0:7] = [0, 0, BODY_Z, 1, 0, 0, 0]
_joint_offset = 7
for i, n in enumerate(LEG_NAMES):
    _d.qpos[_joint_offset + 3 * i + 0] = np.deg2rad(REST_COXA_DEG)
    _d.qpos[_joint_offset + 3 * i + 1] = np.deg2rad(REST_FEMUR_DEG)
    _d.qpos[_joint_offset + 3 * i + 2] = np.deg2rad(REST_TIBIA_DEG)
mujoco.mj_forward(_test_model, _d)

with mujoco.Renderer(_test_model, height=420, width=720) as r:
    r.update_scene(_d, "chase")
    rest_img = r.render()
media.show_image(rest_img)

## 2. The Gymnasium environment

Wraps the MuJoCo model in the standard `env.reset() / env.step(action)` interface that SB3 expects.

- **Action (18 dim, `[-1, 1]`)** — joint position deltas around the rest pose, scaled to ±0.3 rad and written to position actuators.
- **Observation (53 dim)** — chassis z + chassis quaternion + 18 joint angles + 6 chassis velocities + 18 joint velocities + 6 foot-contact flags.
- **Reward** — forward velocity + alive bonus + upright bonus − control cost − jerk cost − lateral drift cost.
- **Termination** — chassis falls below z=0.08 or tilts more than ~60°.

In [ ]:
# 4. The HexapodEnv class.
class HexapodEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 40}

    def __init__(self, xml_string=MODEL_XML, frame_skip=5, max_steps=1000, render_mode=None):
        super().__init__()
        self.xml = xml_string
        self.model = mujoco.MjModel.from_xml_string(self.xml)
        self.data = mujoco.MjData(self.model)
        self.frame_skip = frame_skip
        self.max_steps = max_steps
        self.render_mode = render_mode

        self._leg_names = LEG_NAMES
        self._joint_qpos = {}
        for n in self._leg_names:
            for j in ("coxa", "femur", "tibia"):
                jid = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, f"{n}_{j}_j")
                self._joint_qpos[(n, j)] = self.model.jnt_qposadr[jid]
        self._foot_site_ids = {
            n: mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SITE, f"{n}_foot")
            for n in self._leg_names
        }
        self.chassis_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "chassis")
        self._rest_ctrl = np.array([np.deg2rad(REST_COXA_DEG),
                                    np.deg2rad(REST_FEMUR_DEG),
                                    np.deg2rad(REST_TIBIA_DEG)] * 6, dtype=np.float32)
        assert self._rest_ctrl.shape[0] == self.model.nu

        self.action_space = spaces.Box(-1.0, 1.0, (self.model.nu,), dtype=np.float32)
        obs_dim = 1 + 4 + 18 + 6 + 18 + 6
        self.observation_space = spaces.Box(-np.inf, np.inf, (obs_dim,), dtype=np.float32)
        self._step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[0:7] = [0.0, 0.0, BODY_Z, 1.0, 0.0, 0.0, 0.0]
        for n in self._leg_names:
            self.data.qpos[self._joint_qpos[(n, "coxa")]] = np.deg2rad(REST_COXA_DEG)
            self.data.qpos[self._joint_qpos[(n, "femur")]] = np.deg2rad(REST_FEMUR_DEG)
            self.data.qpos[self._joint_qpos[(n, "tibia")]] = np.deg2rad(REST_TIBIA_DEG)
        self.data.qpos[7:] += self.np_random.uniform(-0.03, 0.03, size=self.model.nq - 7)
        mujoco.mj_forward(self.model, self.data)
        self._step_count = 0
        return self._get_obs(), {}

    def step(self, action):
        action = np.clip(action, -1.0, 1.0).astype(np.float32)
        self.data.ctrl[:] = self._rest_ctrl + ACTION_RANGE_RAD * action
        for _ in range(self.frame_skip):
            mujoco.mj_step(self.model, self.data)
        obs = self._get_obs()
        reward = self._get_reward()
        terminated = self._fell_over()
        self._step_count += 1
        truncated = self._step_count >= self.max_steps
        return obs, reward, terminated, truncated, {}

    def _get_obs(self):
        d = self.data
        return np.concatenate([
            d.qpos[2:3], d.qpos[3:7], d.qpos[7:7+18],
            d.qvel[0:6], d.qvel[6:6+18],
            self._foot_contacts(),
        ]).astype(np.float32)

    def _foot_contacts(self):
        d = self.data
        z = np.array([d.site_xpos[self._foot_site_ids[n]][2] for n in self._leg_names])
        return (z < 0.015).astype(np.float32)

    def _get_reward(self):
        d = self.data
        r_forward = 2.0 * float(d.qvel[0])
        r_alive = 0.5
        upright = float(d.xmat[self.chassis_id].reshape(3, 3)[2, 2])
        r_upright = 0.5 * upright
        c_ctrl = 0.0005 * float(np.square(d.ctrl).sum())
        c_jerk = 0.0001 * float(np.square(d.qvel[6:]).sum())
        c_lateral = 0.5 * float(d.qvel[1] ** 2)
        return r_forward + r_alive + r_upright - c_ctrl - c_jerk - c_lateral

    def _fell_over(self):
        z = float(self.data.qpos[2])
        upright = float(self.data.xmat[self.chassis_id].reshape(3, 3)[2, 2])
        return z < 0.08 or upright < 0.5

_e = HexapodEnv()
_o, _ = _e.reset(seed=0)
print(f"observation dim: {_o.shape[0]} (expected {_e.observation_space.shape[0]})")
print(f"action dim:      {_e.action_space.shape[0]}")

In [ ]:
# 5. Visualize random-action baseline (the spider before learning).
_env = HexapodEnv()
_env.reset(seed=42)
_frames_random = []
with mujoco.Renderer(_env.model, height=360, width=640) as r:
    for _ in range(80):                          # ~2 s at 40 fps
        _env.step(_env.action_space.sample())
        r.update_scene(_env.data, "chase")
        _frames_random.append(r.render().copy())

print("Random-action behavior (before training):")
media.show_video(_frames_random, fps=40)

## 3. Train PPO on the GPU

8 parallel CPU envs feed transitions; PyTorch policy network runs on the GPU. 1.5M total timesteps. Expected wallclock: ~25–35 min on a free T4.

Per-update progress prints to the cell output. To watch live curves, open Tensorboard in another tab — the cell after training plots a static summary either way.

In [ ]:
# 6. Train.
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback

N_ENVS = 8
TOTAL_TIMESTEPS = 1_500_000

def make_env(rank):
    def _init():
        env = HexapodEnv()
        env = Monitor(env)
        env.reset(seed=rank)
        return env
    return _init

if __name__ == "__main__" or True:                # Colab-safe
    vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])
    model = PPO(
        "MlpPolicy", vec_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.0,
        policy_kwargs={"net_arch": [256, 256]},
        tensorboard_log="/content/tb/",
        device=DEVICE,
        verbose=1,
    )
    ckpt = CheckpointCallback(save_freq=200_000 // N_ENVS,
                              save_path="/content/ckpts/",
                              name_prefix="hexapod")
    t0 = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=ckpt, progress_bar=True)
    print(f"\nTraining wallclock: {(time.time()-t0)/60:.1f} min")
    model.save("/content/hexapod_walker")
    vec_env.close()

In [ ]:
# 7. Plot the training reward curve from the Monitor logs.
import glob, csv
rewards, lengths = [], []
for f in sorted(glob.glob("/tmp/sb3_monitor*.csv")) + sorted(glob.glob("*.monitor.csv")):
    with open(f) as fh:
        for i, row in enumerate(csv.reader(fh)):
            if i < 2 or not row: continue
            try:
                rewards.append(float(row[0])); lengths.append(int(row[1]))
            except Exception:
                pass

if rewards:
    plt.figure(figsize=(9, 3.2))
    win = max(50, len(rewards) // 30)
    smoothed = np.convolve(rewards, np.ones(win)/win, mode="valid")
    plt.plot(rewards, alpha=0.25, label="per-episode")
    plt.plot(np.arange(len(smoothed)) + win, smoothed, linewidth=2.0, label=f"moving avg (n={win})")
    plt.xlabel("episode"); plt.ylabel("reward")
    plt.title("PPO training reward"); plt.legend(); plt.grid(True, alpha=0.5)
    plt.tight_layout(); plt.show()
else:
    print("(no monitor logs found - skipping plot)")

## 4. Watch the trained policy walk

Loads the saved model, runs a deterministic rollout, renders the result as a video plus a top-down trajectory plot showing where the chassis went.

In [ ]:
# 8. Roll out the trained policy and render.
eval_env = HexapodEnv(max_steps=400)              # 400 steps × 25 ms = 10 s
model = PPO.load("/content/hexapod_walker", env=eval_env, device=DEVICE)

obs, _ = eval_env.reset(seed=123)
frames, xyz_log = [], []
with mujoco.Renderer(eval_env.model, height=420, width=720) as r:
    for step in range(400):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, term, trunc, _ = eval_env.step(action)
        xyz_log.append(eval_env.data.qpos[0:3].copy())
        r.update_scene(eval_env.data, "chase")
        frames.append(r.render().copy())
        if term or trunc:
            break

xyz_log = np.array(xyz_log)
print(f"Trained-policy rollout: {len(frames)} frames "
      f"({len(frames)*0.025:.2f}s), "
      f"chassis advanced {xyz_log[-1, 0]-xyz_log[0, 0]:+.3f} m in x.")
media.show_video(frames, fps=40)

In [ ]:
# 9. Top-down trajectory of the chassis during the trained rollout.
plt.figure(figsize=(8, 3.4))
plt.plot(xyz_log[:, 0], xyz_log[:, 1], "-", linewidth=2.2, color="#1f5fa8", label="chassis path")
plt.scatter(xyz_log[0, 0], xyz_log[0, 1], s=80, color="#1a7f37", zorder=5, label="start")
plt.scatter(xyz_log[-1, 0], xyz_log[-1, 1], s=80, color="#cf222e", zorder=5, label="end")
plt.axhline(0, linestyle=":", alpha=0.4); plt.axvline(0, linestyle=":", alpha=0.4)
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.gca().set_aspect("equal", adjustable="datalim")
plt.title(f"Trained policy: walked {xyz_log[-1, 0]-xyz_log[0, 0]:+.2f} m forward in {len(frames)*0.025:.1f} s")
plt.legend(); plt.grid(True, alpha=0.5); plt.tight_layout(); plt.show()

## Next steps for students

If the spider learned to walk: try **doubling `TOTAL_TIMESTEPS`** to 3M and compare the final video and trajectory. Then tweak `coxa_amp`-style ideas by changing the reward coefficients (e.g. raise the upright reward to 1.5).

If the spider learned to scoot on its belly or wiggle in place: raise the **upright reward** coefficient from 0.5 to 1.5, or raise the early-termination threshold from `z < 0.08` to `z < 0.12`. See the **Pitfalls** tab of the RL tutorial artifact for symptom→cause→fix mappings.

Advanced: replace SB3+CPU envs with **MJX + Brax** for ~20× faster training by running thousands of envs in parallel directly on the GPU. Same hexapod model, ported to JAX. Same conceptual reward function. Worth as a follow-up notebook once this one is solid.